# 0. Librerías y generalización de rutas

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import re, warnings, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
import seaborn as sns
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROC, FIG = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "reports" / "figuras"
for p in (RAW, PROC, FIG):
    p.mkdir(parents=True, exist_ok=True)

# Zonas con las que se entrena (texto que debe aparecer en la columna Línea). None = todas.
ZONAS = ["AutoNorte", "Calle 26", "NQS Sur"]

# Estaciones de estudio: código -> nombre corto. Los resultados se reportan para estas.
ESTUDIO = {"02200": "Alcalá", "06001": "Modelia", "07504": "Terreros"}

# Festivos del periodo (confirmar con un calendario oficial)
FESTIVOS = pd.to_datetime(["2026-06-08", "2026-06-15", "2026-06-29", "2026-07-13",
                           "2026-07-20", "2026-08-07", "2026-08-17"])

URL_GTFS = "https://storage.googleapis.com/gtfs-estaticos/GTFS_20260712.zip"
DIAS_TEST = 21        # los últimos días del periodo se reservan como conjunto de prueba
N_FOLDS = 5           # validación cruzada por grupos de fecha
RECARGAR = False      # True = volver a leer los Excel aunque exista el archivo procesado

In [ ]:
# Estilo común de las gráficas
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

AZUL, NARANJA, VERDE, GRIS, TINTA = "#2a78d6", "#eb6834", "#1baf7a", "#b9b8b2", "#52514e"
COLOR_DIA = {"habil": AZUL, "sabado": NARANJA, "domingo_festivo": VERDE}
NOMBRE_DIA = {"habil": "Día hábil", "sabado": "Sábado", "domingo_festivo": "Domingo o festivo"}
COLOR_EST = dict(zip(ESTUDIO.values(), [AZUL, NARANJA, VERDE]))
DIAS = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
MAPA_AZUL = LinearSegmentedColormap.from_list("azul", ["#fcfcfb", "#cde2fb", "#6da7ec", "#2a78d6", "#0d366b"])
MAPA_DIVERGENTE = LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRIS,
    "axes.grid": True, "grid.color": "#e7e6e2", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "lines.linewidth": 2, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": TINTA, "xtick.color": TINTA, "ytick.color": TINTA, "legend.frameon": False,
    "axes.prop_cycle": cycler(color=[AZUL, NARANJA, VERDE, "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"])})

def eje_horas(ax):
    ax.set_xticks(range(4, 24, 4)); ax.set_xticklabels([f"{h:02d}:00" for h in range(4, 24, 4)])
    ax.set_xlabel("Hora del día")

def miles(x, _=None):
    return f"{x / 1000:,.0f} mil".replace(",", ".") if abs(x) >= 1000 else f"{x:,.0f}"

# 1. Carga de Datos


## 2. Modelos Utilizados

### 2.1 Partición en entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split

X = data[["linea", "estación", "intervalo", "fecha", "tipo_dia"]]
y = data["validaciones"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=data["estación"], random_state=2026)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print((X_test["estación"].value_counts() / data["estación"].value_counts() * 100).round(1))

### 2.2 Inicialización Modelos


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

def a_hora(v):                       # "05:15" -> 5.25
    h, m = str(v).split(":")[:2]
    return int(h[-2:]) + int(m) / 60

X_train, X_test = X_train.copy(), X_test.copy()
X_train["hora"] = X_train["intervalo"].map(a_hora)
X_test["hora"] = X_test["intervalo"].map(a_hora)

CATEGORICAS, NUMERICAS = ["linea", "estación", "tipo_dia"], ["hora"]

def preparar():
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
                              ("num", StandardScaler(), NUMERICAS)])

### 2.3 Ejecución

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import SplineTransformer
from sklearn.kernel_approximation import Nystroem
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

con_splines = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
                                 ("spl", SplineTransformer(), NUMERICAS)])

modelos = {"Base (media)": make_pipeline(preparar(), DummyRegressor()),
           "OLS":          make_pipeline(preparar(), LinearRegression()),
           "Ridge":        make_pipeline(preparar(), Ridge()),
           "Lasso":        make_pipeline(preparar(), Lasso()),
           "Splines":      make_pipeline(con_splines, LinearRegression()),
           "Kernel (RBF)": make_pipeline(preparar(), Nystroem(random_state=42), Ridge())}

filas = {}
for nombre, m in modelos.items():
    p = m.fit(X_train, y_train).predict(X_test)
    filas[nombre] = {"MAE": mean_absolute_error(y_test, p),
                     "RMSE": root_mean_squared_error(y_test, p),
                     "R2": r2_score(y_test, p)}
display(pd.DataFrame(filas).T.round(3))

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold

GRILLAS = {"OLS":          {},
           "Ridge":        {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
           "Lasso":        {"lasso__alpha": [0.001, 0.01, 0.1, 1, 10]},
           "Splines":      {"columntransformer__spl__n_knots": [5, 10, 20, 40, 60, 80]},
           "Kernel (RBF)": {"nystroem__gamma": [0.03, 0.1, 0.3, 1],
                            "nystroem__n_components": [200],
                            "ridge__alpha": [0.01, 0.1, 1, 10]}}

tabla = pd.DataFrame(filas).T
mejores = tabla.drop(index="Base (media)").sort_values("RMSE").index[:2].tolist()
print("Los dos mejores de la tabla anterior:", mejores)

cv = KFold(n_splits=5, shuffle=True, random_state=2026)
ajustados, comparacion = {}, {}
for nombre in mejores:
    b = GridSearchCV(modelos[nombre], GRILLAS[nombre], cv=cv, n_jobs=-1,
                     scoring="neg_root_mean_squared_error").fit(X_train, y_train)
    ajustados[nombre] = b.best_estimator_
    p = b.predict(X_test)
    comparacion[nombre] = {"RMSE en CV": -b.best_score_,
                           "RMSE test (antes)": tabla.loc[nombre, "RMSE"],
                           "RMSE test (ajustado)": root_mean_squared_error(y_test, p),
                           "MAE test (ajustado)": mean_absolute_error(y_test, p),
                           "R2 test (ajustado)": r2_score(y_test, p)}
    print(f"\n{nombre}: mejores hiperparámetros ->", b.best_params_ or "no tiene")
    for k, v in b.best_params_.items():
        if len(GRILLAS[nombre][k]) > 1 and v in (GRILLAS[nombre][k][0], GRILLAS[nombre][k][-1]):
            print(f"   OJO: {k} = {v} quedó en el borde de la grilla; hay que ampliarla hacia ese lado")

comparacion = pd.DataFrame(comparacion).T.sort_values("RMSE en CV")
display(comparacion.round(3))
print("\nEl mejor por validación cruzada es:", comparacion.index[0])

In [ ]:
mejor = comparacion.index[0]            # el de menor RMSE en validación cruzada
modelo = ajustados[mejor]
fechas = pd.to_datetime(data["fecha"])
lunes = data[(fechas.dt.month == 8) & (fechas.dt.dayofweek == 0)]
lunes = lunes[lunes["tipo_dia"] == lunes["tipo_dia"].mode()[0]]        # quita el lunes festivo
FECHA = str(pd.to_datetime(lunes["fecha"]).drop_duplicates().sample(1, random_state=None).iloc[0].date())
print("Lunes escogido:", FECHA)                    # un miércoles hábil; cambiar para ver otro día
ESTUDIO = ["Alcalá", "Modelia", "Terreros"]

fig, ejes = plt.subplots(1, 3, figsize=(13.5, 3.9))
errores = {}
for ax, est in zip(ejes, ESTUDIO):
    nombre = data.loc[data["estación"].str.contains(est, case=False), "estación"].iloc[0]
    d = data[(data["estación"] == nombre) & (pd.to_datetime(data["fecha"]) == FECHA)].copy()
    d["hora"] = d["intervalo"].map(a_hora)
    d = d.sort_values("hora")
    d["pred"] = modelo.predict(d[X_train.columns])
    en_test = d.index.isin(X_test.index)
    ax.scatter(d.hora[~en_test], d.validaciones[~en_test], s=14, color="#b9b8b2", label="Real (dato de entrenamiento)")
    ax.scatter(d.hora[en_test], d.validaciones[en_test], s=18, color="#1f1f1d", label="Real (dato de prueba)")
    ax.plot(d.hora, d.pred, color="#2a78d6", linewidth=2, label=mejor)
    ax.set_title(f"{est}, {FECHA}", loc="left", fontweight="bold", fontsize=11)
    ax.set_xticks(range(4, 24, 4)); ax.set_xticklabels([f"{h:02d}:00" for h in range(4, 24, 4)])
    ax.set_xlabel("Hora del día")
    ax.spines[["top", "right"]].set_visible(False)

    t = X_test[X_test["estación"] == nombre]              # todas las fechas, solo datos de prueba
    p = modelo.predict(t)
    errores[est] = {"Registros en prueba": len(t), "Entradas promedio": y_test[t.index].mean(),
                    "MAE": mean_absolute_error(y_test[t.index], p),
                    "RMSE": root_mean_squared_error(y_test[t.index], p)}
ejes[0].set_ylabel("Entradas en la franja de 15 min")
fig.legend(*ejes[0].get_legend_handles_labels(), loc="lower center", ncol=3, frameon=False)
plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.show()

errores = pd.DataFrame(errores).T
errores["MAE / promedio"] = errores["MAE"] / errores["Entradas promedio"]
print("Modelo:", mejor, "| Error en los datos de prueba, todas las fechas:")
display(errores.round(2))